In [1]:
# ============================================
# FEATURE ENGINEERING
# CELL 1 — DATASET & FEATURE SCHEMA
# ============================================

from pathlib import Path
import sys

import pandas as pd
import numpy as np

print("Libraries imported successfully")


# Project root
PROJECT_ROOT = Path(
    r"C:\Users\nk134\OneDrive\Desktop\coding\phishguard-x"
)

print("Project root:", PROJECT_ROOT)
print("Exists:", PROJECT_ROOT.exists())


# Dataset
DATA_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "phishguard_features_final_11500.csv"
)

df_fe = pd.read_csv(DATA_FILE)

print("\n========== DATASET ==========")
print("Shape:", df_fe.shape)


# Final model features
FINAL_FEATURES = [
    "Have_IP",
    "Have_At",
    "URL_Length",
    "URL_Depth",
    "Redirection",
    "https_Domain",
    "TinyURL",
    "Prefix/Suffix",
    "DNS_Record",
    "Web_Traffic",
    "Domain_Age",
    "Domain_End",
    "iFrame",
    "Mouse_Over",
    "Web_Forwards",
]


# Required columns
required_columns = [
    "Domain",
    *FINAL_FEATURES,
    "Label"
]

missing_columns = [
    col for col in required_columns
    if col not in df_fe.columns
]

print("\n========== SCHEMA CHECK ==========")

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All required columns are present.")


print("\n========== FINAL MODEL FEATURES ==========")
print("Number of features:", len(FINAL_FEATURES))
print(FINAL_FEATURES)


print("\n========== LABEL DISTRIBUTION ==========")
print(df_fe["Label"].value_counts().sort_index())


print("\n========== DATA TYPES ==========")
print(df_fe[required_columns].dtypes)

Libraries imported successfully
Project root: C:\Users\nk134\OneDrive\Desktop\coding\phishguard-x
Exists: True

========== DATASET ==========
Shape: (11500, 18)

========== SCHEMA CHECK ==========
All required columns are present.

========== FINAL MODEL FEATURES ==========
Number of features: 15
['Have_IP', 'Have_At', 'URL_Length', 'URL_Depth', 'Redirection', 'https_Domain', 'TinyURL', 'Prefix/Suffix', 'DNS_Record', 'Web_Traffic', 'Domain_Age', 'Domain_End', 'iFrame', 'Mouse_Over', 'Web_Forwards']

========== LABEL DISTRIBUTION ==========
Label
0    6500
1    5000
Name: count, dtype: int64

========== DATA TYPES ==========
Domain             str
Have_IP          int64
Have_At          int64
URL_Length       int64
URL_Depth        int64
Redirection      int64
https_Domain     int64
TinyURL          int64
Prefix/Suffix    int64
DNS_Record       int64
Web_Traffic      int64
Domain_Age       int64
Domain_End       int64
iFrame           int64
Mouse_Over       int64
Web_Forwards     int64


In [2]:
# ============================================
# FEATURE ENGINEERING
# CELL 2 — FEATURE VALUE VALIDATION
# ============================================

print("========== FEATURE VALUE VALIDATION ==========")

# Binary features
BINARY_FEATURES = [
    "Have_IP",
    "Have_At",
    "URL_Length",
    "Redirection",
    "https_Domain",
    "TinyURL",
    "Prefix/Suffix",
    "DNS_Record",
    "Web_Traffic",
    "Domain_Age",
    "Domain_End",
    "iFrame",
    "Mouse_Over",
    "Web_Forwards",
]

print("\n========== BINARY FEATURE CHECK ==========")

binary_check = {}

for feature in BINARY_FEATURES:
    unique_values = sorted(df_fe[feature].dropna().unique().tolist())
    binary_check[feature] = unique_values
    print(f"{feature}: {unique_values}")

# URL Depth
print("\n========== URL DEPTH CHECK ==========")

url_depth_values = sorted(
    df_fe["URL_Depth"].dropna().unique().tolist()
)

print("Unique URL_Depth values:", url_depth_values)
print("Minimum:", df_fe["URL_Depth"].min())
print("Maximum:", df_fe["URL_Depth"].max())

# Missing values
print("\n========== MISSING VALUE CHECK ==========")

missing_values = df_fe[required_columns].isnull().sum()

print(missing_values[missing_values > 0])

if missing_values.sum() == 0:
    print("No missing values detected.")

# Validation
invalid_binary_features = []

for feature, values in binary_check.items():
    if not set(values).issubset({0, 1}):
        invalid_binary_features.append(feature)

print("\n========== VALIDATION RESULT ==========")

if invalid_binary_features:
    print("Invalid binary features:", invalid_binary_features)
else:
    print("All binary features contain only 0/1 values.")

if df_fe["URL_Depth"].min() >= 0:
    print("URL_Depth contains no negative values.")
else:
    print("Warning: Negative URL_Depth value detected.")

========== FEATURE VALUE VALIDATION ==========

========== BINARY FEATURE CHECK ==========
Have_IP: [0, 1]
Have_At: [0, 1]
URL_Length: [0, 1]
Redirection: [0, 1]
https_Domain: [0, 1]
TinyURL: [0, 1]
Prefix/Suffix: [0, 1]
DNS_Record: [0, 1]
Web_Traffic: [0, 1]
Domain_Age: [0, 1]
Domain_End: [0, 1]
iFrame: [0, 1]
Mouse_Over: [0, 1]
Web_Forwards: [0, 1]

========== URL DEPTH CHECK ==========
Unique URL_Depth values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 14, 16, 17, 28]
Minimum: 0
Maximum: 28

========== MISSING VALUE CHECK ==========
Series([], dtype: int64)
No missing values detected.

========== VALIDATION RESULT ==========
All binary features contain only 0/1 values.
URL_Depth contains no negative values.


In [3]:
# ============================================
# FEATURE ENGINEERING
# CELL 3 — FEATURE DISTRIBUTION & SKEW CHECK
# ============================================

print("========== FEATURE DISTRIBUTION & SKEW ==========")

numeric_features = FINAL_FEATURES.copy()

feature_stats = pd.DataFrame({
    "Feature": numeric_features,
    "Mean": [df_fe[f].mean() for f in numeric_features],
    "Std": [df_fe[f].std() for f in numeric_features],
    "Min": [df_fe[f].min() for f in numeric_features],
    "Max": [df_fe[f].max() for f in numeric_features],
    "Skewness": [df_fe[f].skew() for f in numeric_features]
})

print(feature_stats.to_string(index=False))

print("\n========== HIGH SKEW FEATURES ==========")

high_skew = feature_stats[
    feature_stats["Skewness"].abs() > 1
]

if high_skew.empty:
    print("No feature has |skewness| > 1.")
else:
    print(high_skew.to_string(index=False))

========== FEATURE DISTRIBUTION & SKEW ==========
      Feature     Mean      Std  Min  Max   Skewness
      Have_IP 0.000783 0.027965    0    1  35.708699
      Have_At 0.004522 0.067095    0    1  14.772113
   URL_Length 0.542870 0.498180    0    1  -0.172134
    URL_Depth 1.930783 2.086254    0   28   1.733223
  Redirection 0.013043 0.113466    0    1   8.584818
 https_Domain 0.000087 0.009325    0    1 107.238053
      TinyURL 0.063217 0.243364    0    1   3.590164
Prefix/Suffix 0.162783 0.369183    0    1   1.827147
   DNS_Record 0.135304 0.342063    0    1   2.132700
  Web_Traffic 0.736783 0.440399    0    1  -1.075498
   Domain_Age 0.336870 0.472660    0    1   0.690385
   Domain_End 0.825565 0.379499    0    1  -1.716060
       iFrame 0.839652 0.366944    0    1  -1.851569
   Mouse_Over 0.228261 0.419730    0    1   1.295054
 Web_Forwards 0.265826 0.441791    0    1   1.060295

========== HIGH SKEW FEATURES ==========
      Feature     Mean      Std  Min  Max   Skewness
      H

In [4]:
# ============================================
# FEATURE ENGINEERING
# CELL 4 — CONSTANT & NEAR-CONSTANT FEATURES
# ============================================

print("========== CONSTANT FEATURE CHECK ==========")

constant_features = []

for feature in FINAL_FEATURES:
    unique_count = df_fe[feature].nunique()

    if unique_count <= 1:
        constant_features.append(feature)

print("Constant features:", constant_features)

print("\n========== NEAR-CONSTANT FEATURE CHECK ==========")

near_constant_features = []

for feature in FINAL_FEATURES:
    value_counts = df_fe[feature].value_counts(normalize=True)

    dominant_percentage = value_counts.iloc[0]

    if dominant_percentage >= 0.99:
        near_constant_features.append(
            (feature, dominant_percentage)
        )

if near_constant_features:
    for feature, percentage in near_constant_features:
        print(
            f"{feature}: "
            f"{percentage * 100:.4f}% dominant value"
        )
else:
    print("No near-constant feature found.")

print("\n========== VALIDATION RESULT ==========")

if constant_features:
    print("Constant features detected:", constant_features)
else:
    print("No constant features found.")

if near_constant_features:
    print("Near-constant features detected.")
else:
    print("No near-constant features detected.")

========== CONSTANT FEATURE CHECK ==========
Constant features: []

========== NEAR-CONSTANT FEATURE CHECK ==========
Have_IP: 99.9217% dominant value
Have_At: 99.5478% dominant value
https_Domain: 99.9913% dominant value

========== VALIDATION RESULT ==========
No constant features found.
Near-constant features detected.


In [5]:
# ============================================
# FEATURE ENGINEERING
# CELL 5 — DUPLICATE FEATURE PATTERNS
# ============================================

print("========== DUPLICATE FEATURE PATTERNS ==========")

feature_matrix = df_fe[FINAL_FEATURES].copy()

duplicate_feature_rows = feature_matrix.duplicated(
    keep=False
)

duplicate_feature_count = duplicate_feature_rows.sum()

unique_feature_patterns = feature_matrix.drop_duplicates().shape[0]

total_rows = len(feature_matrix)

print("Total rows:", total_rows)
print("Unique feature patterns:", unique_feature_patterns)
print("Rows belonging to duplicate feature patterns:",
      duplicate_feature_count)

print("\n========== DUPLICATE PATTERN RATE ==========")

duplicate_pattern_rate = (
    duplicate_feature_count / total_rows
) * 100

print(
    f"Duplicate feature-pattern rate: "
    f"{duplicate_pattern_rate:.2f}%"
)

print("\n========== VALIDATION RESULT ==========")

if duplicate_feature_count == 0:
    print("No duplicate feature patterns found.")
else:
    print(
        "Duplicate feature patterns exist."
        " This will be considered during model validation."
    )

========== DUPLICATE FEATURE PATTERNS ==========
Total rows: 11500
Unique feature patterns: 499
Rows belonging to duplicate feature patterns: 11336

========== DUPLICATE PATTERN RATE ==========
Duplicate feature-pattern rate: 98.57%

========== VALIDATION RESULT ==========
Duplicate feature patterns exist. This will be considered during model validation.


In [6]:
# ============================================
# FEATURE ENGINEERING
# CELL 6 — CLASS-WISE VARIANCE
# ============================================

print("========== CLASS-WISE FEATURE VARIANCE ==========")

variance_table = pd.DataFrame({
    "Feature": FINAL_FEATURES,
    "Legitimate_Variance": [
        df_fe.loc[df_fe["Label"] == 0, feature].var()
        for feature in FINAL_FEATURES
    ],
    "Phishing_Variance": [
        df_fe.loc[df_fe["Label"] == 1, feature].var()
        for feature in FINAL_FEATURES
    ]
})

print(variance_table.to_string(index=False))

print("\n========== ZERO-VARIANCE CHECK WITHIN EACH CLASS ==========")

zero_variance_features = []

for feature in FINAL_FEATURES:
    legitimate_var = df_fe.loc[
        df_fe["Label"] == 0, feature
    ].var()

    phishing_var = df_fe.loc[
        df_fe["Label"] == 1, feature
    ].var()

    if legitimate_var == 0 or phishing_var == 0:
        zero_variance_features.append(feature)

if zero_variance_features:
    print(
        "Zero-variance feature in at least one class:",
        zero_variance_features
    )
else:
    print(
        "No feature has zero variance within either class."
    )

========== CLASS-WISE FEATURE VARIANCE ==========
      Feature  Legitimate_Variance  Phishing_Variance
      Have_IP             0.000000           0.001797
      Have_At             0.003679           0.005570
   URL_Length             0.177542           0.186835
    URL_Depth             5.094480           2.268029
  Redirection             0.012608           0.013223
 https_Domain             0.000000           0.000200
      TinyURL             0.062587           0.054824
Prefix/Suffix             0.020046           0.226881
   DNS_Record             0.093685           0.144534
  Web_Traffic             0.162500           0.224445
   Domain_Age             0.163136           0.249986
   Domain_End             0.181465           0.083226
       iFrame             0.155822           0.103944
   Mouse_Over             0.174782           0.177997
 Web_Forwards             0.205478           0.180234

========== ZERO-VARIANCE CHECK WITHIN EACH CLASS ==========
Zero-variance feature in 

In [7]:
# ============================================
# FEATURE ENGINEERING
# CELL 7 — CORRELATION & REDUNDANCY CHECK
# ============================================

print("========== FEATURE CORRELATION ==========")

correlation_matrix = df_fe[FINAL_FEATURES].corr()

# Get upper triangle only
upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1
    ).astype(bool)
)

correlation_pairs = (
    upper_triangle
    .stack()
    .sort_values(
        key=lambda x: x.abs(),
        ascending=False
    )
)

print("\n========== TOP CORRELATED FEATURE PAIRS ==========")

print(correlation_pairs.head(15).to_string())

print("\n========== HIGH CORRELATION PAIRS ==========")

high_correlation = correlation_pairs[
    correlation_pairs.abs() >= 0.70
]

if high_correlation.empty:
    print("No feature pair has |correlation| >= 0.70.")
else:
    print(high_correlation.to_string())

print("\n========== VALIDATION RESULT ==========")

if high_correlation.empty:
    print(
        "No highly redundant feature pair detected "
        "at the 0.70 threshold."
    )
else:
    print(
        "Highly correlated feature pairs detected."
    )
    print(
        "No features will be removed automatically."
    )

========== FEATURE CORRELATION ==========

========== TOP CORRELATED FEATURE PAIRS ==========
Mouse_Over     Web_Forwards     0.902880
DNS_Record     Mouse_Over       0.727351
               Web_Forwards     0.657393
URL_Length     URL_Depth        0.644544
DNS_Record     Domain_Age       0.555000
               Web_Traffic     -0.400306
Domain_Age     Mouse_Over       0.390884
Web_Traffic    Mouse_Over      -0.370749
Domain_Age     Web_Forwards     0.332414
Web_Traffic    Web_Forwards    -0.328229
Domain_Age     Domain_End       0.327137
Prefix/Suffix  Domain_Age       0.293727
URL_Depth      Prefix/Suffix   -0.288193
URL_Length     Domain_Age      -0.283667
               Prefix/Suffix   -0.277674

========== HIGH CORRELATION PAIRS ==========
Mouse_Over  Web_Forwards    0.902880
DNS_Record  Mouse_Over      0.727351

========== VALIDATION RESULT ==========
Highly correlated feature pairs detected.
No features will be removed automatically.


In [8]:
# ============================================
# FEATURE ENGINEERING
# CELL 8 — DOMAIN-AWARE TRAIN/TEST SPLIT
# ============================================

from sklearn.model_selection import GroupShuffleSplit

print("========== DOMAIN-AWARE TRAIN/TEST SPLIT ==========")

X = df_fe[FINAL_FEATURES].copy()
y = df_fe["Label"].copy()
groups = df_fe["Domain"].copy()

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

print("\n========== SHAPES ==========")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\n========== LABEL DISTRIBUTION ==========")

print("Training:")
print(y_train.value_counts().sort_index())

print("\nTesting:")
print(y_test.value_counts().sort_index())

print("\n========== DOMAIN COUNTS ==========")

train_domains = set(groups_train.unique())
test_domains = set(groups_test.unique())

print("Training domains:", len(train_domains))
print("Testing domains :", len(test_domains))

domain_overlap = train_domains.intersection(test_domains)

print("Domain overlap:", len(domain_overlap))

print("\n========== VALIDATION RESULT ==========")

if len(domain_overlap) == 0:
    print(
        "PASS: No domain appears in both training and test sets."
    )
else:
    print(
        "WARNING: Domain overlap detected:",
        len(domain_overlap)
    )

========== DOMAIN-AWARE TRAIN/TEST SPLIT ==========

========== SHAPES ==========
X_train: (9111, 15)
X_test : (2389, 15)

========== LABEL DISTRIBUTION ==========
Training:
Label
0    5111
1    4000
Name: count, dtype: int64

Testing:
Label
0    1389
1    1000
Name: count, dtype: int64

========== DOMAIN COUNTS ==========
Training domains: 3953
Testing domains : 989
Domain overlap: 0

========== VALIDATION RESULT ==========
PASS: No domain appears in both training and test sets.


In [10]:
# ============================================
# FEATURE ENGINEERING
# CELL 9 — FINAL MATRIX ALIGNMENT CHECK
# ============================================

print("========== FINAL MATRIX ALIGNMENT CHECK ==========")

print("\n========== FEATURE COLUMNS ==========")

print("X_train columns:")
print(X_train.columns.tolist())

print("\nX_test columns:")
print(X_test.columns.tolist())

print("\n========== FEATURE COUNT ==========")

print("X_train feature count:", X_train.shape[1])
print("X_test feature count :", X_test.shape[1])

print("\n========== INDEX ALIGNMENT ==========")

train_alignment = X_train.index.equals(y_train.index)
test_alignment = X_test.index.equals(y_test.index)

print("X_train / y_train aligned:", train_alignment)
print("X_test / y_test aligned:", test_alignment)

print("\n========== MISSING VALUE CHECK ==========")

train_missing = X_train.isnull().sum().sum()
test_missing = X_test.isnull().sum().sum()
y_train_missing = y_train.isnull().sum()
y_test_missing = y_test.isnull().sum()

print("X_train missing values:", train_missing)
print("X_test missing values :", test_missing)
print("y_train missing values:", y_train_missing)
print("y_test missing values :", y_test_missing)

print("\n========== EXPECTED FEATURE CHECK ==========")

features_match = (
    X_train.columns.tolist() == FINAL_FEATURES
    and
    X_test.columns.tolist() == FINAL_FEATURES
)

print("Features match FINAL_FEATURES:", features_match)

print("\n========== VALIDATION RESULT ==========")

if (
    features_match
    and X_train.shape[1] == 15
    and X_test.shape[1] == 15
    and train_alignment
    and test_alignment
    and train_missing == 0
    and test_missing == 0
    and y_train_missing == 0
    and y_test_missing == 0
):
    print("PASS: Train/test feature matrices are correctly aligned.")
else:
    print("WARNING: Alignment or data-quality issue detected.")

========== FINAL MATRIX ALIGNMENT CHECK ==========

========== FEATURE COLUMNS ==========
X_train columns:
['Have_IP', 'Have_At', 'URL_Length', 'URL_Depth', 'Redirection', 'https_Domain', 'TinyURL', 'Prefix/Suffix', 'DNS_Record', 'Web_Traffic', 'Domain_Age', 'Domain_End', 'iFrame', 'Mouse_Over', 'Web_Forwards']

X_test columns:
['Have_IP', 'Have_At', 'URL_Length', 'URL_Depth', 'Redirection', 'https_Domain', 'TinyURL', 'Prefix/Suffix', 'DNS_Record', 'Web_Traffic', 'Domain_Age', 'Domain_End', 'iFrame', 'Mouse_Over', 'Web_Forwards']

========== FEATURE COUNT ==========
X_train feature count: 15
X_test feature count : 15

========== INDEX ALIGNMENT ==========
X_train / y_train aligned: True
X_test / y_test aligned: True

========== MISSING VALUE CHECK ==========
X_train missing values: 0
X_test missing values : 0
y_train missing values: 0
y_test missing values : 0

========== EXPECTED FEATURE CHECK ==========
Features match FINAL_FEATURES: True

========== VALIDATION RESULT ==========
PASS

In [11]:
# ============================================
# FEATURE ENGINEERING
# CELL 10 — FEATURE SCALING
# ============================================

from sklearn.preprocessing import StandardScaler

print("========== FEATURE SCALING ==========")

scaler = StandardScaler()

# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using training statistics
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape :", X_test_scaled.shape)

print("\n========== TRAINING SET STATISTICS ==========")

train_scaled_mean = X_train_scaled.mean(axis=0)
train_scaled_std = X_train_scaled.std(axis=0)

print("Mean range:",
      train_scaled_mean.min(),
      "to",
      train_scaled_mean.max())

print("Std range:",
      train_scaled_std.min(),
      "to",
      train_scaled_std.max())

print("\n========== VALIDATION RESULT ==========")

if (
    X_train_scaled.shape == X_train.shape
    and
    X_test_scaled.shape == X_test.shape
):
    print("PASS: Feature scaling completed successfully.")
    print("Scaler fitted only on training data.")
else:
    print("WARNING: Scaling dimension mismatch detected.")

========== FEATURE SCALING ==========
X_train_scaled shape: (9111, 15)
X_test_scaled shape : (2389, 15)

========== TRAINING SET STATISTICS ==========
Mean range: -1.4544640568462155e-16 to 7.486785493685612e-17
Std range: 0.9999999999999999 to 1.0000000000000002

========== VALIDATION RESULT ==========
PASS: Feature scaling completed successfully.
Scaler fitted only on training data.


In [12]:
# ============================================
# FEATURE ENGINEERING
# CELL 11 — SCALED DATAFRAME CREATION
# ============================================

print("========== SCALED FEATURE DATAFRAMES ==========")

X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=FINAL_FEATURES,
    index=X_train.index
)

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=FINAL_FEATURES,
    index=X_test.index
)

print("X_train_scaled_df shape:",
      X_train_scaled_df.shape)

print("X_test_scaled_df shape :",
      X_test_scaled_df.shape)

print("\n========== COLUMN CHECK ==========")

print(
    "Training columns match:",
    X_train_scaled_df.columns.tolist() == FINAL_FEATURES
)

print(
    "Testing columns match :",
    X_test_scaled_df.columns.tolist() == FINAL_FEATURES
)

print("\n========== INDEX CHECK ==========")

print(
    "Training index preserved:",
    X_train_scaled_df.index.equals(X_train.index)
)

print(
    "Testing index preserved:",
    X_test_scaled_df.index.equals(X_test.index)
)

print("\n========== PREVIEW ==========")

print(X_train_scaled_df.head())

print("\n========== VALIDATION RESULT ==========")

if (
    X_train_scaled_df.shape == X_train.shape
    and
    X_test_scaled_df.shape == X_test.shape
    and
    X_train_scaled_df.columns.tolist() == FINAL_FEATURES
    and
    X_test_scaled_df.columns.tolist() == FINAL_FEATURES
    and
    X_train_scaled_df.index.equals(X_train.index)
    and
    X_test_scaled_df.index.equals(X_test.index)
):
    print("PASS: Scaled DataFrames created correctly.")
else:
    print("WARNING: Scaled DataFrame validation failed.")

========== SCALED FEATURE DATAFRAMES ==========
X_train_scaled_df shape: (9111, 15)
X_test_scaled_df shape : (2389, 15)

========== COLUMN CHECK ==========
Training columns match: True
Testing columns match : True

========== INDEX CHECK ==========
Training index preserved: True
Testing index preserved: True

========== PREVIEW ==========
    Have_IP   Have_At  URL_Length  URL_Depth  Redirection  https_Domain  \
0 -0.023433 -0.066405    0.899741   1.137044    -0.107455     -0.010477   
1 -0.023433 -0.066405    0.899741   0.610106    -0.107455     -0.010477   
3 -0.023433 -0.066405    0.899741   1.663982    -0.107455     -0.010477   
4 -0.023433 -0.066405    0.899741   1.137044    -0.107455     -0.010477   
5 -0.023433 -0.066405    0.899741   2.190921    -0.107455     -0.010477   

    TinyURL  Prefix/Suffix  DNS_Record  Web_Traffic  Domain_Age  Domain_End  \
0 -0.277629      -0.454879   -0.403006     0.619693    1.432929    0.450655   
1 -0.277629      -0.454879   -0.403006     0.61969

In [13]:
# ============================================
# FEATURE ENGINEERING
# CELL 12 — FEATURE-TARGET ASSOCIATION
# ============================================

from scipy.stats import pointbiserialr

print("========== FEATURE-TARGET ASSOCIATION ==========")

association_results = []

for feature in FINAL_FEATURES:
    correlation, p_value = pointbiserialr(
        df_fe[feature],
        df_fe["Label"]
    )

    association_results.append({
        "Feature": feature,
        "Correlation": correlation,
        "P_Value": p_value
    })

association_df = pd.DataFrame(association_results)

association_df["Abs_Correlation"] = (
    association_df["Correlation"].abs()
)

association_df = association_df.sort_values(
    "Abs_Correlation",
    ascending=False
)

print("\n========== FEATURES ORDERED BY ASSOCIATION ==========")

print(
    association_df.to_string(index=False)
)

print("\n========== SIGNIFICANT FEATURES (p < 0.05) ==========")

significant_features = association_df[
    association_df["P_Value"] < 0.05
]

print(
    significant_features[
        ["Feature", "Correlation", "P_Value"]
    ].to_string(index=False)
)

print("\n========== VALIDATION RESULT ==========")

print(
    "Features tested:",
    len(association_df)
)

print(
    "Statistically significant features:",
    len(significant_features)
)

print(
    "\nNote: Statistical significance alone will NOT be used "
    "to remove features."
)

========== FEATURE-TARGET ASSOCIATION ==========

========== FEATURES ORDERED BY ASSOCIATION ==========
      Feature  Correlation       P_Value  Abs_Correlation
   URL_Length    -0.518091  0.000000e+00         0.518091
Prefix/Suffix     0.439560  0.000000e+00         0.439560
    URL_Depth    -0.334560 9.292269e-299         0.334560
   Domain_Age     0.317560 9.266432e-268         0.317560
   Domain_End     0.191447  2.281434e-95         0.191447
  Web_Traffic    -0.152920  4.041989e-61         0.152920
   DNS_Record     0.102298  3.911871e-28         0.102298
       iFrame     0.101701  7.982812e-28         0.101701
 Web_Forwards    -0.059611  1.578410e-10         0.059611
      Have_IP     0.031909  6.207892e-04         0.031909
      TinyURL    -0.018083  5.248611e-02         0.018083
      Have_At     0.014096  1.306623e-01         0.014096
 https_Domain     0.010633  2.542308e-01         0.010633
   Mouse_Over     0.006978  4.543395e-01         0.006978
  Redirection     0.002756

In [14]:
# ============================================
# FEATURE ENGINEERING
# CELL 13 — MUTUAL INFORMATION
# ============================================

from sklearn.feature_selection import mutual_info_classif

print("========== MUTUAL INFORMATION ==========")

mi_scores = mutual_info_classif(
    X_train,
    y_train,
    random_state=42
)

mi_df = pd.DataFrame({
    "Feature": FINAL_FEATURES,
    "Mutual_Information": mi_scores
})

mi_df = mi_df.sort_values(
    "Mutual_Information",
    ascending=False
)

print("\n========== FEATURES ORDERED BY MUTUAL INFORMATION ==========")

print(
    mi_df.to_string(index=False)
)

print("\n========== MUTUAL INFORMATION SUMMARY ==========")

print(
    "Highest MI feature:",
    mi_df.iloc[0]["Feature"]
)

print(
    "Highest MI score:",
    mi_df.iloc[0]["Mutual_Information"]
)

print("\n========== VALIDATION RESULT ==========")

if len(mi_df) == len(FINAL_FEATURES):
    print(
        "PASS: Mutual information calculated "
        "for all 15 features."
    )
else:
    print(
        "WARNING: Mutual information calculation mismatch."
    )

print(
    "\nNote: Mutual information is used as supporting "
    "feature-selection evidence, not as an automatic removal rule."
)

========== MUTUAL INFORMATION ==========

========== FEATURES ORDERED BY MUTUAL INFORMATION ==========
      Feature  Mutual_Information
   URL_Length            0.137246
    URL_Depth            0.124923
Prefix/Suffix            0.113148
   Domain_Age            0.024951
       iFrame            0.019102
   Domain_End            0.016565
  Web_Traffic            0.013741
 https_Domain            0.004749
   DNS_Record            0.002498
  Redirection            0.001160
      Have_At            0.000653
      TinyURL            0.000000
      Have_IP            0.000000
   Mouse_Over            0.000000
 Web_Forwards            0.000000

========== MUTUAL INFORMATION SUMMARY ==========
Highest MI feature: URL_Length
Highest MI score: 0.1372456262183157

========== VALIDATION RESULT ==========
PASS: Mutual information calculated for all 15 features.

Note: Mutual information is used as supporting feature-selection evidence, not as an automatic removal rule.


In [15]:
# ============================================
# FEATURE ENGINEERING
# CELL 14 — FEATURE SELECTION EVIDENCE SUMMARY
# ============================================

print("========== FEATURE SELECTION EVIDENCE SUMMARY ==========")

# Correlation with target
target_corr = (
    df_fe[FINAL_FEATURES + ["Label"]]
    .corr(numeric_only=True)["Label"]
    .drop("Label")
)

# Mutual information
mi_lookup = mi_df.set_index("Feature")["Mutual_Information"]

# Combine evidence
feature_evidence = pd.DataFrame({
    "Feature": FINAL_FEATURES,
    "Target_Correlation": [
        target_corr[feature]
        for feature in FINAL_FEATURES
    ],
    "Mutual_Information": [
        mi_lookup[feature]
        for feature in FINAL_FEATURES
    ]
})

feature_evidence["Abs_Target_Correlation"] = (
    feature_evidence["Target_Correlation"].abs()
)

feature_evidence = feature_evidence.sort_values(
    "Abs_Target_Correlation",
    ascending=False
)

print("\n========== COMBINED FEATURE EVIDENCE ==========")

print(
    feature_evidence.to_string(index=False)
)

print("\n========== STRONGER SIGNALS ==========")

strong_features = feature_evidence[
    (feature_evidence["Abs_Target_Correlation"] >= 0.20) |
    (feature_evidence["Mutual_Information"] >= 0.02)
]

print(
    strong_features[
        [
            "Feature",
            "Target_Correlation",
            "Mutual_Information"
        ]
    ].to_string(index=False)
)

print("\n========== FEATURE SELECTION DECISION ==========")

print(
    "No feature is removed automatically based on "
    "correlation or mutual information alone."
)

print(
    "Final feature set remains:",
    len(FINAL_FEATURES),
    "features."
)

print("\n========== VALIDATION RESULT ==========")

if len(feature_evidence) == len(FINAL_FEATURES):
    print(
        "PASS: Evidence summary generated for all 15 features."
    )
else:
    print(
        "WARNING: Feature evidence count mismatch."
    )

========== FEATURE SELECTION EVIDENCE SUMMARY ==========

========== COMBINED FEATURE EVIDENCE ==========
      Feature  Target_Correlation  Mutual_Information  Abs_Target_Correlation
   URL_Length           -0.518091            0.137246                0.518091
Prefix/Suffix            0.439560            0.113148                0.439560
    URL_Depth           -0.334560            0.124923                0.334560
   Domain_Age            0.317560            0.024951                0.317560
   Domain_End            0.191447            0.016565                0.191447
  Web_Traffic           -0.152920            0.013741                0.152920
   DNS_Record            0.102298            0.002498                0.102298
       iFrame            0.101701            0.019102                0.101701
 Web_Forwards           -0.059611            0.000000                0.059611
      Have_IP            0.031909            0.000000                0.031909
      TinyURL           -0.018083   

In [16]:
# ============================================
# FEATURE ENGINEERING
# CELL 15 — SCALING VERIFICATION
# ============================================

print("========== SCALING VERIFICATION ==========")

# Verify training-set statistics after StandardScaler
train_means = X_train_scaled_df.mean()
train_stds = X_train_scaled_df.std(ddof=0)

print("\n========== TRAINING SCALED STATISTICS ==========")

print("Mean range:")
print(
    "Minimum:", train_means.min(),
    "Maximum:", train_means.max()
)

print("\nStandard deviation range:")
print(
    "Minimum:", train_stds.min(),
    "Maximum:", train_stds.max()
)

# Verify test set was transformed using the same scaler
print("\n========== SCALER INFORMATION ==========")

print("Scaler type:", type(scaler).__name__)
print("Number of features:", scaler.n_features_in_)

print("\n========== FEATURE ALIGNMENT ==========")

print(
    "Scaler feature count:",
    scaler.n_features_in_
)

print(
    "Expected feature count:",
    len(FINAL_FEATURES)
)

print(
    "Feature columns match:",
    list(X_train_scaled_df.columns) == FINAL_FEATURES
)

print("\n========== VALIDATION RESULT ==========")

if (
    scaler.n_features_in_ == len(FINAL_FEATURES)
    and
    np.allclose(train_means.values, 0, atol=1e-10)
    and
    np.allclose(train_stds.values, 1, atol=1e-10)
    and
    list(X_train_scaled_df.columns) == FINAL_FEATURES
):
    print("PASS: Feature scaling is correctly applied.")
    print("PASS: Scaler was fitted using the training data.")
    print("PASS: All 15 features remain aligned.")
else:
    print("WARNING: Scaling verification failed.")

========== SCALING VERIFICATION ==========

========== TRAINING SCALED STATISTICS ==========
Mean range:
Minimum: -1.4544640568462155e-16 Maximum: 7.486785493685612e-17

Standard deviation range:
Minimum: 0.9999999999999999 Maximum: 1.0000000000000002

========== SCALER INFORMATION ==========
Scaler type: StandardScaler
Number of features: 15

========== FEATURE ALIGNMENT ==========
Scaler feature count: 15
Expected feature count: 15
Feature columns match: True

========== VALIDATION RESULT ==========
PASS: Feature scaling is correctly applied.
PASS: Scaler was fitted using the training data.
PASS: All 15 features remain aligned.


In [17]:
# ============================================
# FEATURE ENGINEERING
# CELL 16 — SCALED DATA INTEGRITY CHECK
# ============================================

print("========== SCALED DATA INTEGRITY CHECK ==========")

print("\n========== SHAPES ==========")

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("X_train_scaled:", X_train_scaled_df.shape)
print("X_test_scaled:", X_test_scaled_df.shape)

print("\n========== INDEX ALIGNMENT ==========")

print(
    "Train index alignment:",
    X_train_scaled_df.index.equals(X_train.index)
)

print(
    "Test index alignment:",
    X_test_scaled_df.index.equals(X_test.index)
)

print("\n========== COLUMN ALIGNMENT ==========")

print(
    "Train columns match:",
    list(X_train_scaled_df.columns) == FINAL_FEATURES
)

print(
    "Test columns match:",
    list(X_test_scaled_df.columns) == FINAL_FEATURES
)

print("\n========== MISSING VALUES ==========")

print(
    "Train missing values:",
    X_train_scaled_df.isnull().sum().sum()
)

print(
    "Test missing values:",
    X_test_scaled_df.isnull().sum().sum()
)

print("\n========== FINITE VALUE CHECK ==========")

print(
    "Train all finite:",
    np.isfinite(X_train_scaled_df.to_numpy()).all()
)

print(
    "Test all finite:",
    np.isfinite(X_test_scaled_df.to_numpy()).all()
)

print("\n========== VALIDATION RESULT ==========")

if (
    X_train_scaled_df.shape == X_train.shape
    and
    X_test_scaled_df.shape == X_test.shape
    and
    X_train_scaled_df.index.equals(X_train.index)
    and
    X_test_scaled_df.index.equals(X_test.index)
    and
    list(X_train_scaled_df.columns) == FINAL_FEATURES
    and
    list(X_test_scaled_df.columns) == FINAL_FEATURES
    and
    X_train_scaled_df.isnull().sum().sum() == 0
    and
    X_test_scaled_df.isnull().sum().sum() == 0
    and
    np.isfinite(X_train_scaled_df.to_numpy()).all()
    and
    np.isfinite(X_test_scaled_df.to_numpy()).all()
):
    print("PASS: Scaled feature matrices are fully valid.")
else:
    print("WARNING: Scaled data integrity check failed.")

========== SCALED DATA INTEGRITY CHECK ==========

========== SHAPES ==========
X_train: (9111, 15)
X_test: (2389, 15)
X_train_scaled: (9111, 15)
X_test_scaled: (2389, 15)

========== INDEX ALIGNMENT ==========
Train index alignment: True
Test index alignment: True

========== COLUMN ALIGNMENT ==========
Train columns match: True
Test columns match: True

========== MISSING VALUES ==========
Train missing values: 0
Test missing values: 0

========== FINITE VALUE CHECK ==========
Train all finite: True
Test all finite: True

========== VALIDATION RESULT ==========
PASS: Scaled feature matrices are fully valid.


In [18]:
# ============================================
# FEATURE ENGINEERING
# CELL 17 — FEATURE SELECTION STABILITY
# ============================================

print("========== FEATURE SELECTION STABILITY ==========")

# Rank features by absolute target correlation
corr_rank = (
    feature_evidence
    .sort_values("Abs_Target_Correlation", ascending=False)
    ["Feature"]
    .tolist()
)

# Rank features by mutual information
mi_rank = (
    mi_df
    .sort_values("Mutual_Information", ascending=False)
    ["Feature"]
    .tolist()
)

print("\n========== TOP 5 — TARGET CORRELATION ==========")

for i, feature in enumerate(corr_rank[:5], start=1):
    print(f"{i}. {feature}")

print("\n========== TOP 5 — MUTUAL INFORMATION ==========")

for i, feature in enumerate(mi_rank[:5], start=1):
    print(f"{i}. {feature}")

# Common top features
top_corr = set(corr_rank[:5])
top_mi = set(mi_rank[:5])

common_features = top_corr.intersection(top_mi)

print("\n========== COMMON TOP FEATURES ==========")

for feature in corr_rank:
    if feature in common_features:
        print(feature)

print("\nNumber of common top-5 features:", len(common_features))

print("\n========== VALIDATION RESULT ==========")

if len(corr_rank) == len(FINAL_FEATURES) and len(mi_rank) == len(FINAL_FEATURES):
    print(
        "PASS: Feature rankings available for all 15 features."
    )
else:
    print(
        "WARNING: Feature ranking count mismatch."
    )

print(
    "\nDecision: No feature is removed solely because "
    "its ranking is low in one method."
)

========== FEATURE SELECTION STABILITY ==========

========== TOP 5 — TARGET CORRELATION ==========
1. URL_Length
2. Prefix/Suffix
3. URL_Depth
4. Domain_Age
5. Domain_End

========== TOP 5 — MUTUAL INFORMATION ==========
1. URL_Length
2. URL_Depth
3. Prefix/Suffix
4. Domain_Age
5. iFrame

========== COMMON TOP FEATURES ==========
URL_Length
Prefix/Suffix
URL_Depth
Domain_Age

Number of common top-5 features: 4

========== VALIDATION RESULT ==========
PASS: Feature rankings available for all 15 features.

Decision: No feature is removed solely because its ranking is low in one method.


In [19]:
# ============================================
# FEATURE ENGINEERING
# CELL 18 — FINAL FEATURE-SET DECISION
# ============================================

print("========== FINAL FEATURE-SET DECISION ==========")

print("\nFinal model features:")
for i, feature in enumerate(FINAL_FEATURES, start=1):
    print(f"{i}. {feature}")

print("\n========== FEATURE COUNT ==========")
print("Final feature count:", len(FINAL_FEATURES))

print("\n========== REMOVED FEATURE ==========")
print("Right_Click")
print("Reason: Constant feature with only one observed value.")

print("\n========== FEATURE SELECTION EVIDENCE ==========")

print(
    "Top features consistently identified by "
    "correlation and mutual information:"
)

for feature in common_features:
    print("-", feature)

print("\n========== FINAL DECISION ==========")

print(
    "All 15 FINAL_FEATURES are retained for model training."
)

print(
    "No additional feature is removed solely on the basis "
    "of correlation, variance, or mutual information."
)

print("\n========== VALIDATION RESULT ==========")

required_final_features = set(FINAL_FEATURES)

actual_final_features = set(
    X_train.columns
)

if (
    len(FINAL_FEATURES) == 15
    and
    required_final_features == actual_final_features
    and
    "Right_Click" not in required_final_features
    and
    "Label" not in required_final_features
    and
    "Domain" not in required_final_features
):
    print(
        "PASS: Final 15-feature model matrix is correctly defined."
    )
else:
    print(
        "WARNING: Final feature-set validation failed."
    )

========== FINAL FEATURE-SET DECISION ==========

Final model features:
1. Have_IP
2. Have_At
3. URL_Length
4. URL_Depth
5. Redirection
6. https_Domain
7. TinyURL
8. Prefix/Suffix
9. DNS_Record
10. Web_Traffic
11. Domain_Age
12. Domain_End
13. iFrame
14. Mouse_Over
15. Web_Forwards

========== FEATURE COUNT ==========
Final feature count: 15

========== REMOVED FEATURE ==========
Right_Click
Reason: Constant feature with only one observed value.

========== FEATURE SELECTION EVIDENCE ==========
Top features consistently identified by correlation and mutual information:
- Prefix/Suffix
- URL_Depth
- Domain_Age
- URL_Length

========== FINAL DECISION ==========
All 15 FINAL_FEATURES are retained for model training.
No additional feature is removed solely on the basis of correlation, variance, or mutual information.

========== VALIDATION RESULT ==========
PASS: Final 15-feature model matrix is correctly defined.
